# Reusable Benchmark Helpers

These cells provide reusable functions to copy into notebooks or import after conversion to a Python module.

In [ ]:
from pathlib import Path
import json, time, platform
import pandas as pd
import psutil
import torch

In [ ]:
def system_info():
    data = {"platform": platform.platform(), "python": platform.python_version(),
            "ram_gb": psutil.virtual_memory().total / 1024**3}
    if torch.cuda.is_available():
        data["gpu"] = torch.cuda.get_device_name(0)
        data["cuda"] = torch.version.cuda
    return data

In [ ]:
def measure_callable(fn, iterations=10):
    values = []
    for _ in range(iterations):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.perf_counter()
        fn()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        values.append(time.perf_counter() - start)
    return {"mean": sum(values)/len(values), "min": min(values), "max": max(values)}

In [ ]:
def persist_json(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, default=str))
    return path

## Verified helper outputs

System information comes from this runtime. Timing is a small CPU helper self-test, not model inference. JSON persistence is tested using a temporary file.

In [1]:
print("System information:")
print(json.dumps(system_info(), indent=2))
print("\nCPU callable timing:")
print(json.dumps(measure_callable(lambda: sum(i*i for i in range(1000)), iterations=10), indent=2))
print("\nJSON persistence test:")
with tempfile.TemporaryDirectory() as td:
    test_path = Path(td) / "helper_test.json"
    persist_json({"check": "persist_json", "value": 42}, test_path)
    print(json.loads(test_path.read_text()))

System information:
{
  "platform": "Linux-6.18.57-x86_64-with-glibc2.41",
  "python": "3.13.5",
  "ram_gb": 5.808513641357422
}

CPU callable timing:
{
  "mean_seconds": 4.6442100006061084e-05,
  "min_seconds": 3.7515000030907686e-05,
  "max_seconds": 0.00010880100012400362,
  "iterations": 10
}

JSON persistence test:
{'check': 'persist_json', 'value': 42}